In [1]:
from chi import server, context, lease, network
import chi, os, time, datetime

context.version = "1.0"
context.choose_project()
context.choose_site(default="KVM@TACC")

In [3]:
LEASE_NAME = "proj27-mlops-devops"

l = lease.Lease(
    LEASE_NAME,
    duration=datetime.timedelta(hours=8),
)
l.add_flavor_reservation(
    id=chi.server.get_flavor_id("m1.large"),
    amount=1,
)
l.submit(idempotent=True)
l.show()

Waiting for lease to start...


Lease proj27-mlops-devops has reached status active


HTML(value='\n        <h2>Lease Details</h2>\n        <table>\n            <tr><th>Name</th><td>proj27-mlops-d…

Lease Details:
Name: proj27-mlops-devops
ID: 671154c9-7c99-4ce0-9cf1-580b67e98182
Status: ACTIVE
Start Date: 2026-04-03 00:00:00
End Date: 2026-04-03 08:00:00
User ID: 4eade18a2ae0994011ddd200e76ed5f53af817042cb649ace44d56f6194a51eb
Project ID: 89f528973fea4b3a981f9b2344e522de

Node Reservations:

Floating IP Reservations:

Network Reservations:

Flavor Reservations:
ID: 44600261-3938-421d-82da-154534922185, Status: active, Flavor: 44600261-3938-421d-82da-154534922185, Amount: 1

Events:


In [4]:
SERVER_NAME = "proj27-k8s-node"

s = server.Server(
    SERVER_NAME,
    image_name="CC-Ubuntu24.04",
    flavor_name=l.get_reserved_flavors()[0].name,
)
s.submit(idempotent=True)

The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.


Waiting for server proj27-k8s-node's status to become ACTIVE. This typically takes 10 minutes for baremetal, but can take up to 20 minutes.


Server has moved to status ACTIVE


Attribute,proj27-k8s-node
Id,3766b492-dbf2-490d-9e0a-b5bbd73dea2e
Status,ACTIVE
Image Name,CC-Ubuntu24.04
Flavor Name,reservation:44600261-3938-421d-82da-154534922185
Addresses,sharednet1: IP: 10.56.2.192 (v4) Type: fixed MAC: fa:16:3e:96:3e:5f
Network Name,sharednet1
Created At,2026-04-03T00:01:43Z
Keypair,hw3655_nyu_edu-jupyter
Reservation Id,None
Host Id,8e0d56804f84ab945a9e08b1513858ec87e1b8180724abcd725ec42b


In [5]:
s.refresh()
s.show(type="widget")

Attribute,proj27-k8s-node
Id,3766b492-dbf2-490d-9e0a-b5bbd73dea2e
Status,ACTIVE
Image Name,CC-Ubuntu24.04
Flavor Name,reservation:44600261-3938-421d-82da-154534922185
Addresses,sharednet1: IP: 10.56.2.192 (v4) Type: fixed MAC: fa:16:3e:96:3e:5f
Network Name,sharednet1
Created At,2026-04-03T00:01:43Z
Keypair,hw3655_nyu_edu-jupyter
Reservation Id,None
Host Id,8e0d56804f84ab945a9e08b1513858ec87e1b8180724abcd725ec42b


In [6]:
s.associate_floating_ip()

The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.


'129.114.27.109'

In [9]:
s.refresh()
s.check_connectivity()
s.show(type="widget")

Checking connectivity to 129.114.27.109 port 22.


Connection successful


Attribute,proj27-k8s-node
Id,3766b492-dbf2-490d-9e0a-b5bbd73dea2e
Status,ACTIVE
Image Name,CC-Ubuntu24.04
Flavor Name,reservation:44600261-3938-421d-82da-154534922185
Addresses,sharednet1: IP: 10.56.2.192 (v4) Type: fixed MAC: fa:16:3e:96:3e:5f IP: 129.114.27.109 (v4) Type: floating MAC: fa:16:3e:96:3e:5f
Network Name,sharednet1
Created At,2026-04-03T00:01:43Z
Keypair,hw3655_nyu_edu-jupyter
Reservation Id,None
Host Id,8e0d56804f84ab945a9e08b1513858ec87e1b8180724abcd725ec42b


Attribute,proj27-k8s-node
Id,3766b492-dbf2-490d-9e0a-b5bbd73dea2e
Status,ACTIVE
Image Name,CC-Ubuntu24.04
Flavor Name,reservation:44600261-3938-421d-82da-154534922185
Addresses,sharednet1: IP: 10.56.2.192 (v4) Type: fixed MAC: fa:16:3e:96:3e:5f IP: 129.114.27.109 (v4) Type: floating MAC: fa:16:3e:96:3e:5f
Network Name,sharednet1
Created At,2026-04-03T00:01:43Z
Keypair,hw3655_nyu_edu-jupyter
Reservation Id,None
Host Id,8e0d56804f84ab945a9e08b1513858ec87e1b8180724abcd725ec42b


In [35]:
security_groups = [
  {'name': "allow-ssh", "protocol": "tcp", 'port': 22, 'description': "Enable SSH traffic on TCP port 22"},
  {'name': "allow-8888", "protocol": "tcp", 'port': 8888, 'description': "Enable TCP port 8888 (used by Jupyter)"},
  {"name": "allow-30080", "protocol": "tcp", "port": 30080, "description": "Enable Jitsi Web on TCP port 30080"},
  {"name": "allow-jvb-10000-udp", "port": 10000, "protocol": "udp", "description": "Enable Jitsi JVB media on UDP port 10000"},
  {"name": "allow-jvb-30000-udp", "port": 30000, "protocol": "udp", "description": "Enable Jitsi JVB media on UDP port 30000"},
  {"name": "allow-30500", "port": 30500, "protocol": "tcp", "description": "Enable MLflow UI on TCP port 30500"},
]

In [36]:
for sg in security_groups:
  secgroup = network.SecurityGroup({
      'name': sg['name'],
      'description': sg['description'],
  })
  secgroup.add_rule(direction='ingress', protocol=sg["protocol"], port=sg['port'])
  secgroup.submit(idempotent=True)
  s.add_security_group(sg['name'])

print(f"updated security groups: {[sg['name'] for sg in security_groups]}")

The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.


updated security groups: ['allow-30500']


In [34]:
s.refresh()
s.check_connectivity()

Checking connectivity to 129.114.27.109 port 22.


Connection successful


Use local terminal to ssh:
```bash
ssh -i ~/.ssh/id_rsa_chameleon cc@129.114.27.109
```